---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 📊 LDA & QDA
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> When the response is binary and predictors are approximately Gaussian, LDA and QDA exploit that distributional structure to build classifiers that outperform logistic regression — especially with small samples.

### Dataset: Default (ISLP)
10,000 credit card customers — predict whether a customer will default on their payment.
Same dataset as the Logistic Regression notebook so you can directly compare methods.

### What you'll learn
- How LDA derives its decision threshold analytically from Bayes' theorem
- LDA vs QDA: shared vs class-specific covariance
- When LDA beats logistic regression (small n, Gaussian predictors)
- Multiclass LDA and QDA (K > 2 classes)
- ROC and AUC comparison across all four classifiers

### Time: ~50 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# ── Load Default dataset (ISLP Chapter 4) ────────────────────────────────────
try:
    default_df = pd.read_csv('https://www.statlearning.com/s/Default.csv')
    print(f'✓ Default.csv (ISLP): {default_df.shape}')
except Exception:
    default_df = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Default.csv')
    print(f'✓ Default.csv (fallback): {default_df.shape}')

default_df['default_num'] = (default_df['default'] == 'Yes').astype(int)
default_df['student_num'] = (default_df['student'] == 'Yes').astype(int)

print(f'  Default rate: {default_df["default_num"].mean():.2%}')
print(f'  Students:     {default_df["student_num"].mean():.2%}')

# Feature matrix and target
features = ['balance', 'income', 'student_num']
X = default_df[features].values.astype(float)
y = default_df['default_num'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print(f'  Train: {X_tr.shape}  Test: {X_te.shape}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — How LDA Works: Threshold from Bayes' Theorem

LDA assigns observation x to class k that maximises the **linear discriminant score**:

```
δₖ(x) = xᵀΣ⁻¹μₖ − ½μₖᵀΣ⁻¹μₖ + log(πₖ)
```

where **μₖ** = class mean vector, **Σ** = shared covariance matrix, **πₖ** = class prior.

The **decision boundary** is where δ₁(x) = δ₂(x). In 1D this gives:

```
threshold = (μ₁ + μ₂)/2  −  [σ²/(μ₁−μ₂)] · log(π₁/π₂)
```

- **Equal priors** → threshold = midpoint between class means
- **Unequal priors** → threshold shifts toward the minority class
- The threshold is **not tuned** — it falls directly out of the probability model

In [ ]:
# ── LDA threshold derivation — 1D illustration using balance ─────────────────
# Use balance alone first so we can visualise in 1D
X_1d = default_df[['balance']].values
y_1d = default_df['default_num'].values

lda_1d = LinearDiscriminantAnalysis().fit(X_1d, y_1d)

mu0    = X_1d[y_1d==0].mean()
mu1    = X_1d[y_1d==1].mean()
pi0    = (y_1d==0).mean()
pi1    = (y_1d==1).mean()
sigma2 = np.var(X_1d)

threshold_mid   = (mu0 + mu1) / 2
threshold_prior = (mu0 + mu1) / 2 - (sigma2 / (mu1 - mu0)) * np.log(pi1 / pi0)

print("LDA threshold derivation (1D: balance predicts default)")
print(f"  μ(No Default) = ${mu0:,.2f}   μ(Default) = ${mu1:,.2f}")
print(f"  π(No Default) = {pi0:.4f}   π(Default) = {pi1:.4f}")
print()
print(f"  Midpoint (equal priors):    ${threshold_mid:,.2f}")
print(f"  Actual threshold (priors):  ${threshold_prior:,.2f}")
print(f"  Shift due to unequal priors: ${threshold_prior - threshold_mid:+,.2f}")

# Posterior probability curve
x_range = np.linspace(0, 3000, 400)
log_ratio = ((mu1 - mu0) / sigma2 * x_range
             - (mu1**2 - mu0**2) / (2 * sigma2)
             + np.log(pi1 / pi0))
p_default = 1 / (1 + np.exp(-log_ratio))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: class distributions
for cls, color, label in [(0,'#1e5fa8','No Default'), (1,'#e85d20','Default')]:
    xi = X_1d[y_1d==cls].ravel()
    axes[0].hist(xi, bins=60, density=True, alpha=0.5, color=color, label=label)
    axes[0].axvline(xi.mean(), color=color, lw=2, ls='--', alpha=0.9)

axes[0].axvline(threshold_prior, color='#1a7a45', lw=2.5,
                label=f'LDA threshold ${threshold_prior:,.0f}')
axes[0].axvline(threshold_mid,   color='grey', lw=1.5, ls=':',
                label=f'Midpoint ${threshold_mid:,.0f}')
axes[0].set_xlabel('Credit Card Balance ($)')
axes[0].set_ylabel('Density')
axes[0].set_title('Balance Distributions by Default Status\n'
                   'Prior correction shifts threshold from midpoint')
axes[0].legend(fontsize=8)

# Right: posterior probability
axes[1].plot(x_range, p_default, color='#1e5fa8', lw=2.5,
             label='P(Default | balance) — LDA')
axes[1].axhline(0.5, color='grey', lw=1, ls='--', label='Threshold 0.5')
axes[1].axvline(threshold_prior, color='#1a7a45', lw=2,
                label=f'Decision boundary ${threshold_prior:,.0f}')
axes[1].set_xlabel('Credit Card Balance ($)')
axes[1].set_ylabel('P(Default | balance)')
axes[1].set_title('LDA Posterior Probability\n'
                   'Linear log-odds when classes share covariance Σ')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print()
print("📌 The threshold is NOT arbitrary — Bayes' theorem derives it.")
print("   Default rate is only 3.33%, so the prior heavily favours No Default.")
print(f"   This shifts the threshold ${threshold_prior-threshold_mid:+.0f} above the midpoint.")
print("   Customers need a quite high balance before LDA predicts default.")


## 🔬 Part 2 — Fitting LDA and QDA on the Default Dataset

**LDA** assumes all classes share the same covariance matrix Σ → linear boundary.
**QDA** estimates a separate covariance Σₖ per class → quadratic boundary.

QDA is more flexible but needs more data: p(p+1)/2 parameters per class vs p for LDA.
With p=3 and only ~333 defaults in the training set, QDA may overfit.

In [ ]:
# ── Fit LDA, QDA, Logistic Regression — compare all three ────────────────────
models = {
    'LDA':                  LinearDiscriminantAnalysis(),
    'QDA':                  QuadraticDiscriminantAnalysis(),
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
        'acc':  model.score(X_te, y_te),
        'auc':  roc_auc_score(y_te, y_prob)
    }

# Summary table
print(f"{'Method':22s}  {'Accuracy':>10}  {'AUC-ROC':>9}")
print("-" * 47)
for name, r in results.items():
    print(f"{name:22s}  {r['acc']:>10.4f}  {r['auc']:>9.4f}")

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_te, r['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['No Default','Default'])        .plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{name}\nAcc={r['acc']:.3f}  AUC={r['auc']:.3f}")

plt.suptitle('LDA vs QDA vs Logistic — Default Dataset', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# ROC curves
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#1e5fa8', '#e85d20', '#1a7a45']
for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_te, r['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f"{name} (AUC={r['auc']:.3f})")
ax.plot([0,1],[0,1], 'k--', lw=1, label='Random (AUC=0.5)')
ax.fill_between(*roc_curve(y_te, results['LDA']['y_prob'])[:2],
                alpha=0.05, color='#1e5fa8')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — LDA, QDA, Logistic Regression')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print()
print("📌 All three methods produce similar AUC on this dataset.")
print("   balance is the dominant predictor — the linear boundary is sufficient.")
print("   QDA offers little gain: defaults and non-defaults have similar spread.")


## 📊 Part 3 — LDA Covariance Assumption: When It Matters

LDA assumes **classes share the same covariance matrix Σ** (equal spread in all directions).
QDA relaxes this — each class gets its own Σₖ.

Visualising the class covariance structures reveals whether the LDA assumption is reasonable.

In [ ]:
# ── Visualise class covariance — do defaults and non-defaults have same spread?
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
feat_pairs = [('balance','income'), ('balance','student_num'), ('income','student_num')]

for ax, (f1, f2) in zip(axes, feat_pairs):
    i1, i2 = features.index(f1), features.index(f2)
    for cls, color, label, marker in [
            (0,'#1e5fa8','No Default','o'), (1,'#e85d20','Default','D')]:
        mask = y_tr == cls
        ax.scatter(X_tr[mask, i1], X_tr[mask, i2],
                   color=color, alpha=0.2, s=8, marker=marker, label=label)
        # Draw 1-sigma ellipse
        cov  = np.cov(X_tr[mask, i1], X_tr[mask, i2])
        mean = X_tr[mask][:, [i1, i2]].mean(axis=0)
        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:, order]
        theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        from matplotlib.patches import Ellipse
        ell = Ellipse(xy=mean, width=2*np.sqrt(vals[0]),
                      height=2*np.sqrt(vals[1]), angle=theta,
                      color=color, alpha=0.3, lw=2, fill=False)
        ax.add_patch(ell)
    ax.set_xlabel(f1); ax.set_ylabel(f2)
    ax.set_title(f'{f1} vs {f2}')
    if f1 == 'balance': ax.legend(fontsize=8)

plt.suptitle('Class Covariance Structure — Default Dataset\n'
             'Similar ellipse shapes = LDA assumption plausible',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

# Covariance matrices
cov0 = np.cov(X_tr[y_tr==0].T)
cov1 = np.cov(X_tr[y_tr==1].T)
cov_shared = ((y_tr==0).sum() * cov0 + (y_tr==1).sum() * cov1) / len(y_tr)

print("Covariance matrix diagonal (variances) per class:")
print(f"{'Feature':12s}  {'No Default':>12}  {'Default':>10}  {'Ratio':>7}")
print("-" * 48)
for j, fname in enumerate(features):
    r = cov1[j,j] / cov0[j,j]
    flag = ' ← different' if r > 2 or r < 0.5 else ''
    print(f"{fname:12s}  {cov0[j,j]:>12.2f}  {cov1[j,j]:>10.2f}  {r:>7.2f}{flag}")

print()
print("📌 If variance ratios are close to 1.0 → LDA assumption holds.")
print("   Large ratios (>2 or <0.5) → QDA may fit better.")
print("   balance variance differs slightly; LDA is still a good approximation here.")


The ellipses represent 1-sigma (standard deviation) contours of the data distribution for each class.
They are derived from the mean and covariance matrix of the data points for the selected features.

The axes of the ellipse align with the principal components of the covariance matrix,and their lengths are proportional to the standard deviation along those components, visualizing the spread and correlation of the data.

## ✅ Part 4 — When to Use LDA over Logistic Regression

**LDA is preferred when:**
- Predictors are approximately normal within each class
- **Sample size is small** relative to the number of predictors (n close to p)
- Classes are well-separated (logistic regression becomes numerically unstable)

**Why normality + small n favours LDA:**
Logistic regression estimates β by maximum likelihood — needs large n to converge.
LDA estimates μₖ and Σ directly — fewer parameters, stable with small samples.
When normality holds, LDA is the **statistically optimal** linear classifier.

| Situation | Recommended |
|-----------|-------------|
| Large n, normality not assumed | Logistic Regression |
| Small n, normality plausible | **LDA** |
| Class-specific covariance structures | **QDA** |
| p > n | Regularised LDA or penalised logistic |

In [ ]:
# ── Empirical comparison: LDA vs Logistic at different sample sizes ──────────
from sklearn.datasets import make_classification

np.random.seed(42)
n_test = 3000
X_big, y_big = make_classification(
    n_samples=n_test + 800, n_features=8, n_informative=5,
    n_redundant=2, random_state=42)
X_te_fixed = X_big[:n_test];  y_te_fixed = y_big[:n_test]
X_pool     = X_big[n_test:];  y_pool     = y_big[n_test:]

sample_sizes = [20, 30, 50, 75, 100, 200, 500, 800]
lda_accs, lr_accs = [], []

for n in sample_sizes:
    idx = np.random.choice(len(X_pool), n, replace=False)
    Xn, yn = X_pool[idx], y_pool[idx]
    lda_accs.append(LinearDiscriminantAnalysis().fit(Xn,yn).score(X_te_fixed, y_te_fixed))
    try:
        lr_accs.append(LogisticRegression(max_iter=2000).fit(Xn,yn).score(X_te_fixed, y_te_fixed))
    except Exception:
        lr_accs.append(np.nan)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sample_sizes, lda_accs, 'o-', color='#1e5fa8', lw=2.5,
        markersize=7, label='LDA')
ax.plot(sample_sizes, lr_accs,  's--', color='#e85d20', lw=2.5,
        markersize=7, label='Logistic Regression')
ax.axvline(60, color='grey', lw=1, ls=':', alpha=0.7,
           label='LDA advantage zone (n < 60)')
ax.set_xlabel('Training sample size (n)')
ax.set_ylabel('Test accuracy (n_test=3,000)')
ax.set_title('LDA vs Logistic Regression — Small Sample Advantage\n'
             '(Gaussian classes — LDA assumption met)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"{'n':>6}  {'LDA':>8}  {'LogReg':>8}  {'Winner'}")
print("-" * 36)
for n, la, lra in zip(sample_sizes, lda_accs, lr_accs):
    winner = 'LDA ✓' if la > lra else 'LogReg ✓' if lra > la else 'tie'
    print(f"{n:>6}  {la:>8.3f}  {lra:>8.3f}  {winner}")

print()
print("📌 LDA wins at small n when class distributions are Gaussian.")
print("   As n grows both methods converge — they learn the same linear boundary.")


## 🔍 Part 5 — Multiclass (K > 2): LDA and QDA Naturally Extend

LDA computes **K discriminant functions** — one per class — and assigns to the highest.
With K classes, LDA always produces **K−1 discriminant dimensions** for visualisation.

**Multinomial logistic regression** is the alternative when normality is doubtful.
Here we demonstrate on the Default dataset extended to 3 classes (No Default / Low Risk / High Risk) and on Iris for a clean 3-class visual.

In [ ]:
# ── 3-class example: create Low/High risk tiers from Default data ─────────────
# Segment defaults into low-balance and high-balance default for 3 classes
df3 = default_df.copy()
median_bal_default = df3.loc[df3['default_num']==1, 'balance'].median()
df3['risk'] = 0  # No Default
df3.loc[(df3['default_num']==1) & (df3['balance'] < median_bal_default), 'risk'] = 1  # Low-balance default
df3.loc[(df3['default_num']==1) & (df3['balance'] >= median_bal_default), 'risk'] = 2 # High-balance default

X3 = df3[['balance','income','student_num']].values.astype(float)
y3 = df3['risk'].values
class_names = ['No Default', 'Low Risk Default', 'High Risk Default']

X3_tr, X3_te, y3_tr, y3_te = train_test_split(
    X3, y3, test_size=0.3, random_state=42, stratify=y3)

models_mc = {
    'LDA':  LinearDiscriminantAnalysis(),
    'QDA':  QuadraticDiscriminantAnalysis(),
    'Multinomial Logistic': LogisticRegression(
        multi_class='multinomial', max_iter=1000, random_state=42),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
print(f"{'Method':25s}  {'Accuracy':>10}")
print("-" * 40)
for ax, (name, model) in zip(axes, models_mc.items()):
    model.fit(X3_tr, y3_tr)
    y3_pred = model.predict(X3_te)
    acc = model.score(X3_te, y3_te)
    print(f"{name:25s}  {acc:>10.4f}")
    cm = confusion_matrix(y3_te, y3_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Def','Low Risk','High Risk'])        .plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc = {acc:.3f}')
    ax.tick_params(axis='x', labelrotation=30)

plt.suptitle('Multiclass: No Default / Low Risk / High Risk Default',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

# LDA 2D discriminant projection (K-1 = 2 dimensions)
lda_mc = LinearDiscriminantAnalysis(n_components=2).fit(X3_tr, y3_tr)
Z_tr   = lda_mc.transform(X3_tr)
Z_te   = lda_mc.transform(X3_te)

fig, ax = plt.subplots(figsize=(7, 5))
colors3 = ['#1e5fa8','#e85d20','#1a7a45']
for k, (name, color) in enumerate(zip(class_names, colors3)):
    mask = y3_tr == k
    ax.scatter(Z_tr[mask,0], Z_tr[mask,1], color=color, alpha=0.4,
               s=10, label=name)
ax.set_xlabel('LD1 — primary separation axis')
ax.set_ylabel('LD2 — secondary separation axis')
ax.set_title('LDA Discriminant Projection (2D)\n'
             'K=3 classes → K−1=2 discriminant dimensions')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print()
print("📌 LDA projects the 3-feature space onto 2 discriminant axes.")
print("   LD1 separates No Default from both default groups (balance effect).")
print("   LD2 separates Low-balance from High-balance defaults.")
print("   K classes always give K-1 discriminant dimensions.")


## 💼 Exercise

**Task 1 — Threshold adjustment:** Using the LDA model from Part 2, change the classification threshold from 0.5 to 0.2 (to catch more defaults). How many additional true defaults are caught? How many extra false alarms? Is this worthwhile for a bank?

**Task 2 — Normality check:** Use `scipy.stats.normaltest` to test whether `balance` and `income` are normally distributed within each class. Do the results support using LDA over QDA?

**Task 3 — Feature impact:** Refit LDA using only `balance` (1 feature), then `balance + income` (2 features), then all 3. Report test AUC at each step. Which feature adds the most predictive power?

In [ ]:
# ── Exercise workspace ───────────────────────────────────────────────────────
# Variables: default_df, X_tr, X_te, y_tr, y_te, features, results
# results dict has: 'LDA', 'QDA', 'Logistic Regression' keys
# Each has: model, y_pred, y_prob, acc, auc

# Task 1 — Threshold adjustment for LDA
lda_probs = results['LDA']['y_prob']
for threshold in [0.5, 0.3, 0.2, 0.1]:
    y_t = (lda_probs >= threshold).astype(int)
    tp  = ((y_t==1) & (y_te==1)).sum()
    fp  = ((y_t==1) & (y_te==0)).sum()
    fn  = ((y_t==0) & (y_te==1)).sum()
    print(f"Threshold {threshold:.1f}: caught {tp}/{(y_te==1).sum()} defaults "
          f"| false alarms: {fp} | missed: {fn}")

print()

# Task 2 — Normality test
from scipy import stats
print("Normality test (p < 0.05 = not normal):")
for fname, j in zip(['balance','income'], [0,1]):
    for cls, label in [(0,'No Default'),(1,'Default')]:
        stat, p = stats.normaltest(X_tr[y_tr==cls, j])
        print(f"  {fname:8s} | {label:12s}: p = {p:.4f} "
              f"{'→ NOT normal' if p < 0.05 else '→ approx normal'}")
print()

# Task 3 — Feature importance via AUC
print("AUC by feature set:")
for feat_set, idx in [
        (['balance'],              [0]),
        (['balance','income'],     [0,1]),
        (['balance','income','student_num'], [0,1,2])]:
    lda_f = LinearDiscriminantAnalysis().fit(X_tr[:,idx], y_tr)
    auc_f = roc_auc_score(y_te, lda_f.predict_proba(X_te[:,idx])[:,1])
    print(f"  {str(feat_set):40s}: AUC = {auc_f:.4f}")


In [ ]:
# @title 📝 Quiz — LDA & QDA
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** What is the key assumption LDA makes that QDA relaxes?
# @markdown **Q2:** Why does the LDA decision threshold shift away from the midpoint when class priors are unequal?
# @markdown **Q3:** You have n=25 training samples and p=10 Gaussian predictors. LDA or Logistic Regression?
# @markdown **Q4:** With K=4 classes, how many discriminant dimensions does LDA produce?
# @markdown **Q5:** Your balance covariance for defaults is 4x larger than for non-defaults. LDA or QDA?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the submission cell for AI feedback")


In [ ]:
_NB_NAME_ = "LDA & QDA"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {str(v).strip() or '(no answer)'}"
                    for i,(_, v) in enumerate(answers.items(), 1))
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip(): print(f"Student: @{GITHUB_USERNAME.strip()}")
    print(); print(qa); print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why")
    print("  3. Give the ideal complete answer")
    print("  4. If wrong/partial, say which Part to revisit")
    print("\nEnd with: Overall grade + short study plan.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*